# Dietary inference

Estimate whether menu items are **likely vegetarian** or **likely vegan** from the text already available for retrieval. These are discovery hints, not verified restaurant claims. They must not be used for allergen safety or strict dietary filtering.

## Load the normalized menu and completed descriptions

The ingestion notebook supplies `menu`. The existing enrichment cache supplies descriptions without calling an AI provider again.

In [ ]:
%run ./01_menu_api_ingestion.ipynb

from pathlib import Path

from enrichment_cache import EnrichmentCache
from menu_descriptions import load_cached_descriptions, needs_description

enrichment_path = Path("enrichment") / f"{menu.id}.json"
enrichment_cache = EnrichmentCache.load(enrichment_path, menu.id)
description_results = load_cached_descriptions(menu, enrichment_cache, require_all=True)
description_by_item = {result.item_id: result.text for result in description_results}

print(f"Loaded {len(menu.items)} items and {len(description_results)} inferred descriptions")

## Build small, traceable model inputs

Each input contains only the item name, categories, description, and configurable choices. The input hash makes cached results stale whenever this evidence changes.

In [ ]:
import hashlib
import json

category_names = {category.id: category.name for category in menu.categories}

def dietary_input(item):
    description = item.description
    if needs_description(item):
        description = description_by_item.get(item.id)

    return {
        "item_id": item.id,
        "name": item.name,
        "categories": [category_names[id] for id in item.category_ids],
        "description": description,
        "choices": [
            f"{group.name}: {', '.join(choice.name for choice in group.choices)}"
            for group in item.variation_groups
        ],
    }

def input_hash(record):
    text = json.dumps(record, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

dietary_inputs = [dietary_input(item) for item in menu.items]
dietary_inputs[:2]

## Infer likely vegetarian and vegan suitability

The model uses three labels: `likely`, `unlikely`, and `unknown`. `Unknown` is the correct answer when the text is ambiguous. Results are saved after every batch so the notebook can resume safely.

In [ ]:
import os
import time

from dotenv import load_dotenv
from ai_adapters import create_ai_adapter
from menu_dietary import infer_dietary_batch

adapter = create_ai_adapter()
MODEL_ID = adapter.model_id
PROMPT_VERSION = "dietary-v1"
BATCH_SIZE = 25
VALID_LABELS = {"likely", "unlikely", "unknown"}
dietary_cache_path = Path("enrichment") / f"{menu.id}.dietary.json"

if dietary_cache_path.exists():
    dietary_cache = json.loads(dietary_cache_path.read_text(encoding="utf-8"))
    if dietary_cache.get("menu_id") != menu.id:
        raise ValueError("dietary cache belongs to a different menu")
else:
    dietary_cache = {"menu_id": menu.id, "results": {}}

def save_cache():
    dietary_cache_path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = dietary_cache_path.with_suffix(".tmp")
    temporary_path.write_text(
        json.dumps(dietary_cache, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )
    temporary_path.replace(dietary_cache_path)

def current_cached_result(record):
    result = dietary_cache["results"].get(record["item_id"])
    if not isinstance(result, dict):
        return None
    if result.get("input_hash") != input_hash(record):
        return None
    if result.get("model_id") not in adapter.cache_model_ids or result.get("prompt_version") != PROMPT_VERSION:
        return None
    return result

pending = [record for record in dietary_inputs if current_cached_result(record) is None]
print(f"{len(pending)} items need dietary inference")

In [ ]:
if pending:
    for start in range(0, len(pending), BATCH_SIZE):
        batch = pending[start:start + BATCH_SIZE]
        rows = infer_dietary_batch(adapter, batch)
        rows_by_id = {row["item_id"]: row for row in rows}

        for record in batch:
            row = rows_by_id.get(record["item_id"])
            if row is None:
                raise ValueError(f"model omitted item {record['item_id']!r}")
            if row["vegetarian"] not in VALID_LABELS or row["vegan"] not in VALID_LABELS:
                raise ValueError(f"model returned an invalid label for {record['item_id']!r}")
            dietary_cache["results"][record["item_id"]] = {
                "vegetarian": row["vegetarian"],
                "vegan": row["vegan"],
                "reason": row["reason"].strip(),
                "origin": "inferred",
                "model_id": MODEL_ID,
                "prompt_version": PROMPT_VERSION,
                "input_hash": input_hash(record),
            }

        save_cache()
        print(f"Inferred {min(start + BATCH_SIZE, len(pending))}/{len(pending)} pending items")
        if start + BATCH_SIZE < len(pending):
            time.sleep(6)

## Inspect the results

Review likely classifications before using them for ranking. The canonical menu's verified `dietary_attributes` remain unchanged.

In [ ]:
dietary_results = dietary_cache["results"]
item_by_id = {item.id: item for item in menu.items}

for item_id, result in list(dietary_results.items())[:20]:
    print(
        item_by_id[item_id].name,
        f"vegetarian={result['vegetarian']}",
        f"vegan={result['vegan']}",
        f"— {result['reason']}",
    )

print(f"Cached dietary suggestions for {len(dietary_results)} items")

In [ ]:
from collections import Counter

print("Vegetarian:", Counter(row["vegetarian"] for row in dietary_results.values()))
print("Vegan:", Counter(row["vegan"] for row in dietary_results.values()))